# Parent RAG — *Os Sertões*

Parent RAG retrieves small child chunks for precision, but sends their larger parent documents to the language model. This preserves the surrounding context of each retrieved passage.

## 1. Imports and OpenAI API key

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / '.env')
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY was not found in the .env file.')

## 2. Load the PDF and define parent/child chunk sizes

In [ ]:
PDF_PATH = PROJECT_ROOT / 'data' / 'os-sertoes.pdf'

documents = PyPDFLoader(PDF_PATH).load()

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=250,
    separators=['\n\n', '\n', '. ', ' ', '']
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=['\n\n', '\n', '. ', ' ', '']
)

print(f'Loaded pages: {len(documents)}')

## 3. Create the Parent RAG retriever

This step calls the OpenAI embeddings API. The child chunks are stored in ChromaDB, while parent documents are kept in memory for the current notebook session.

In [ ]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(
    collection_name='parent_rag_os_sertoes',
    embedding_function=embeddings
)

docstore = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(documents)
print('Parent RAG retriever created successfully.')

## 4. Question-answering chain

In [ ]:
TEMPLATE = """
You are an assistant specialized in the book 'Os Sertões', by Euclides da Cunha.

Answer in Portuguese using only the provided context.
If the context is insufficient, clearly state that the information was not found.

Question: {question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_template(TEMPLATE)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
qa_chain = load_qa_chain(llm, chain_type='stuff', prompt=prompt)

def ask(question):
    context = retriever.get_relevant_documents(question)
    result = qa_chain.invoke({
        'input_documents': context,
        'question': question
    })
    return result['output_text'], context

## 5. Evaluation questions

In [ ]:
questions = [
    'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?',
    'Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?',
    'Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?',
    'Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?',
    'Quais são os principais aspectos da crítica social e política presentes em Os Sertões? Como esses aspectos refletem a visão do autor sobre o Brasil da época?'
]

for index, question in enumerate(questions, start=1):
    answer, context = ask(question)
    pages = sorted({document.metadata.get('page', 0) + 1 for document in context})
    print(f'Question {index}: {question}')
    print(f'Answer: {answer}')
    print(f'Retrieved pages: {pages}')
    print('-' * 100)